# Model selection

* 입력: train 내부 nested CV의 저장 결과
* 비교: XGBoost·LightGBM × uplift 학습기 7종 × 피처 구성 2종 = 28개 후보
* 실행 범위: 저장 결과 확인; 후보별 학습 재실행 제외

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
candidates = [p for base in (Path.cwd(), *Path.cwd().parents) for p in (base, base / "code")]
CODE = next(p for p in candidates if (p / "evidence").is_dir() and (p / "modeling").is_dir())
PROJECT = CODE.parent

ranking = pd.read_csv(CODE / "evidence/selection/qini_table.csv")
selection = json.loads((CODE / "evidence/selection/selection_result.json").read_text())
assert len(ranking) == ranking.candidate.nunique() == 28
print(f"후보 {len(ranking)}개 · 학습 기록 {selection['n_studies']}개 · 선정 {selection['selected']}")
display(ranking.sort_values(["rank", "candidate"])[["rank", "candidate", "mean", "sd"]].head(10).rename(columns={"mean": "CV 평균 Qini", "sd": "fold 간 SD"}))

후보 28개 · 학습 기록 140개 · 선정 full__xgboost__r


,rank,candidate,CV 평균 Qini,fold 간 SD
3,1,full__xgboost__r,0.011443,0.003839
2,2,full__xgboost__x,0.011386,0.002069
11,3,full__lightgbm__dr,0.011314,0.003822
10,4,full__lightgbm__r,0.011070,0.003434
4,5,full__xgboost__dr,0.010941,0.003341
14,6,compact__xgboost__s,0.010903,0.004653
7,7,full__lightgbm__s,0.010889,0.002968
20,8,compact__xgboost__lai,0.010867,0.003226
9,9,full__lightgbm__x,0.010769,0.002276
0,10,full__xgboost__s,0.010742,0.004064


## 후보 구성·학습 실행 (선택)

* 후보: 엔진 2종 × uplift 학습기 7종 × 피처 구성 2종
* 선택: outer 5 fold 평균 Qini; 각 outer fold의 inner 3 fold에서 40회 튜닝
* 전체 재실행: `data.ipynb`에서 작업 폴더 생성 후 `X5_RUN_FULL=1`


In [ ]:
import ast
import itertools
import os
import subprocess
import sys

def source_constant(filename, name):
    tree = ast.parse((CODE / "modeling" / filename).read_text())
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == name for t in node.targets):
            return ast.literal_eval(node.value)
    raise KeyError(name)

engines = source_constant("learners.py", "ENGINES")
learners = source_constant("learners.py", "LEARNERS")
tracks = source_constant("tune.py", "TRACKS")
candidates = [f"{track}__{engine}__{learner}"
              for engine, learner, track in itertools.product(engines, learners, tracks)]
assert len(candidates) == len(set(candidates)) == 28
assert set(candidates) == set(ranking.candidate)
display(pd.DataFrame({"엔진": engines, "학습기 수": [len(learners)] * len(engines),
                      "피처 구성 수": [len(tracks)] * len(engines)}))
print(f"후보 {len(candidates)}개 · outer 평가 {len(candidates) * 5}개 · 후보·fold당 Optuna 40회")

if os.environ.get("X5_RUN_FULL") == "1":
    assert os.environ.get("X5_WORK_DIR")
    work = Path(os.environ["X5_WORK_DIR"]).resolve()
    assert (work / "data/splits/final_folds.parquet").is_file()
    log = work / "logs/model_selection.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    from concurrent.futures import ThreadPoolExecutor, as_completed

    nested = work / "results/nested"
    studies = nested / "studies"
    storage = nested / "optuna"
    for folder in (studies, storage, nested / "logs"):
        folder.mkdir(parents=True, exist_ok=True)

    def fit_outer(engine, learner, track, fold):
        study = f"{track}__{engine}__{learner}__outer{fold}"
        result = studies / f"{study}.json"
        if result.is_file() and json.loads(result.read_text()).get("n_trials", {}).get("complete") == 40:
            return study, "기존 완료"
        command = [sys.executable, str(work / "modeling/tune.py"),
                   "--engine", engine, "--learner", learner, "--track", track,
                   "--outer", str(fold), "--n-trials", "40",
                   "--storage-dir", str(storage), "--out-dir", str(studies)]
        with (nested / "logs" / f"{study}.log").open("a") as output:
            subprocess.run(command, cwd=work, stdout=output, stderr=subprocess.STDOUT, check=True)
        assert json.loads(result.read_text())["n_trials"]["complete"] == 40
        return study, "완료"

    jobs = [(e, l, t, k) for e, l, t, k in itertools.product(engines, learners, tracks, range(5))]
    with ThreadPoolExecutor(max_workers=6) as pool:
        statuses = [future.result() for future in as_completed(pool.submit(fit_outer, *job) for job in jobs)]
    print("outer 학습:", len(statuses), "개 완료")
    with log.open("w") as output:
        subprocess.run([sys.executable, str(work / "modeling/selection.py"),
                        "--studies-dir", str(studies), "--out-dir", "results/selection"],
                       cwd=work, stdout=output, stderr=subprocess.STDOUT, check=True)
    local = json.loads((work / "results/selection/selection_result.json").read_text())
    print("전체 학습 완료 · 선정:", local["selected"], "· 기록:", log)
else:
    print("전체 학습 실행: 생략 · 저장된 선정 결과 확인")


## 선정 기준 확인

* 기준: train outer 5 fold의 평균 Qini
* 주의: 상위 후보의 평균 차이가 작아 일관된 우월성의 근거로 해석 불가

In [2]:
ordered = ranking.sort_values(["rank", "candidate"]).reset_index(drop=True)
chosen = ranking.set_index("candidate").loc[selection["selected"]]
assert int(chosen["rank"]) == 1
print(f"선정 모델: {selection['selected']}")
print(f"평균 Qini: {chosen['mean']:.6f} · SD: {chosen['sd']:.6f}")
print(f"1·2위 평균 Qini 차이: {ordered.loc[0, 'mean'] - ordered.loc[1, 'mean']:.6f}")
print("후보 비교 입력: train 내부 결과만 사용")

선정 모델: full__xgboost__r
평균 Qini: 0.011443 · SD: 0.003839
1·2위 평균 Qini 차이: 0.000058
후보 비교 입력: train 내부 결과만 사용
